# Комбинация 6: Network + Plan-Execute — техподдержка инцидентов

Планировщик создаёт план расследования инцидента, а агенты-исполнители (`log_agent`, `dba_agent`, `dev_agent`) могут передавать задачи друг другу напрямую через `Command(goto=...)` — без возврата к планировщику, если контекст подсказывает, что следующий шаг очевиден. Стратегия централизована (планировщик), тактика — на месте (агенты).

```mermaid
graph LR
    START((START)) --> planner{{"🎯 Planner<br/><i>составляет план</i>"}}
    planner -->|"route_planner()"| log_agent(["🔹 Log Agent<br/><i>выполняет задачу</i>"])
    planner -->|"resolution есть"| END((END))
    log_agent -->|"Command(goto='dba_agent')"| dba_agent(["🔹 DBA Agent<br/><i>выполняет задачу</i>"])
    log_agent -->|"Command(goto='dev_agent')"| dev_agent(["🔹 Dev Agent<br/><i>выполняет задачу</i>"])
    log_agent -->|"Command(goto='planner')"| planner
    dba_agent -->|"Command(goto='planner')"| planner
    dev_agent -->|"Command(goto='planner')"| planner

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class planner coordinator
    class log_agent,dba_agent,dev_agent worker
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated, Literal

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние инцидента

`State` хранит описание инцидента, план расследования, журнал действий агентов и итоговое решение. Поле `log` использует редьюсер `operator.add` — каждый агент добавляет записи в журнал, не перезаписывая предыдущие.

In [3]:
llm = get_llm()

class State(TypedDict):
    incident: str
    plan: list[str]
    log: Annotated[list[str], operator.add]
    current_step: str
    resolution: str

## Планировщик

Планировщик — центральный узел, задающий стратегию расследования. Он анализирует инцидент и журнал уже выполненных действий, а затем возвращает JSON с планом из 2–3 шагов и именем первого агента. При ошибке парсинга JSON срабатывает fallback с дефолтным планом.

In [4]:
def planner(state: State) -> dict:
    context = "\n".join(state["log"]) if state["log"] else "Начало работы."
    response = llm.invoke([
        SystemMessage(content="""Ты — планировщик инцидентов. Создай план из 2-3 шагов.
Доступные агенты: log_agent (логи), dba_agent (БД), dev_agent (код).
Верни JSON: {"plan": ["шаг1", "шаг2"], "first": "log_agent"}"""),
        HumanMessage(content=f"Инцидент: {state['incident']}\n\nЖурнал:\n{context}")
    ])
    try:
        parsed = json.loads(response.content.strip())
        plan = parsed.get("plan", ["Проверить логи", "Исправить проблему"])
        first = parsed.get("first", "log_agent")
    except json.JSONDecodeError:
        plan = ["Проверить логи", "Исправить проблему"]
        first = "log_agent"
    print(f"  [Планировщик] План: {plan}, первый: {first}")
    return {"plan": plan, "current_step": plan[0] if plan else "done"}

## Агенты-исполнители с Command

Ключевая особенность комбинации Network + Plan-Execute — агенты сами решают, кому передать управление через `Command(goto=...)`. LogAgent анализирует логи инцидента: если проблема в базе данных — он передаёт задачу напрямую `dba_agent`, если в коде — `dev_agent`, без возврата к планировщику. Это быстрее, и контекст не теряется.

DBA и Dev агенты после выполнения работы возвращают управление планировщику для реплана.

In [5]:
def log_agent(state: State) -> Command[Literal["planner", "dba_agent", "dev_agent"]]:
    response = llm.invoke([
        SystemMessage(content="""Ты — агент по логам. Проанализируй инцидент.
Если проблема в БД — передай dba_agent (ответь: HANDOFF dba_agent).
Если в коде — передай dev_agent (ответь: HANDOFF dev_agent).
Иначе — верни результат планировщику."""),
        HumanMessage(content=f"Инцидент: {state['incident']}\n\nШаг: {state['current_step']}")
    ])
    content = response.content
    entry = f"[log_agent]: {content[:100]}"

    if "HANDOFF DBA_AGENT" in content.upper():
        print(f"  [log_agent] → handoff → dba_agent")
        return Command(goto="dba_agent", update={"log": [entry]})
    if "HANDOFF DEV_AGENT" in content.upper():
        print(f"  [log_agent] → handoff → dev_agent")
        return Command(goto="dev_agent", update={"log": [entry]})

    print(f"  [log_agent] → реплан")
    return Command(goto="planner", update={"log": [entry]})


def dba_agent(state: State) -> Command[Literal["planner"]]:
    response = llm.invoke([
        SystemMessage(content="Ты — DBA. Реши проблему с базой данных. Кратко: 2-3 предложения."),
        HumanMessage(content=f"Инцидент: {state['incident']}")
    ])
    print(f"  [dba_agent] Решено → реплан")
    return Command(goto="planner", update={
        "log": [f"[dba_agent]: {response.content[:100]}"],
        "resolution": response.content,
    })


def dev_agent(state: State) -> Command[Literal["planner"]]:
    response = llm.invoke([
        SystemMessage(content="Ты — разработчик. Исправь баг в коде. Кратко: 2-3 предложения."),
        HumanMessage(content=f"Инцидент: {state['incident']}")
    ])
    print(f"  [dev_agent] Решено → реплан")
    return Command(goto="planner", update={
        "log": [f"[dev_agent]: {response.content[:100]}"],
        "resolution": response.content,
    })

## Маршрутизация и сборка графа

Условное ребро из планировщика направляет к `log_agent` — точке входа в сеть агентов. Дальнейшая маршрутизация происходит через `Command` внутри самих агентов. Два условия завершения: наличие `resolution` (проблема решена) или достижение лимита в 5 записей журнала (защита от зацикливания). Fallback-ребро `planner → END` — страховка на случай ошибки маршрутизации.

In [6]:
def route_planner(state: State) -> str:
    if state.get("resolution"):
        return END
    if len(state["log"]) >= 5:
        return END
    return "log_agent"

graph = StateGraph(State)
graph.add_node("planner", planner)
graph.add_node("log_agent", log_agent)
graph.add_node("dba_agent", dba_agent)
graph.add_node("dev_agent", dev_agent)

graph.add_edge(START, "planner")
graph.add_conditional_edges("planner", route_planner, {
    "log_agent": "log_agent", END: END,
})
graph.add_edge("planner", END)  # fallback

app = graph.compile()

## Запуск

Инцидент: API отвечает 500, в логах — ошибка подключения к PostgreSQL. Типичный сценарий: планировщик создаёт план, `log_agent` анализирует логи и обнаруживает проблему с БД, передаёт задачу напрямую `dba_agent` через `Command(goto="dba_agent")`, DBA решает проблему и возвращает управление планировщику для реплана. Планировщик видит `resolution` и завершает работу.

In [7]:
result = app.invoke({
    "incident": "API отвечает 500 на запросы /users — в логах ошибка подключения к PostgreSQL",
    "plan": [], "log": [], "current_step": "", "resolution": "",
})

print(f"\nЖурнал ({len(result['log'])} записей):")
for entry in result["log"]:
    print(f"  {entry}")
print(f"\nРешение: {result.get('resolution', 'Нет')[:200]}")

  [Планировщик] План: ['Проверить логи API и PostgreSQL на признаки отказа соединения, таймаутов и ошибок аутентификации.', 'Если в логах подтверждается проблема с БД, проверить состояние PostgreSQL, доступность хоста, пул соединений и параметры подключения.', 'Если проблема выглядит кодовой или конфигурационной, проверить недавние изменения в коде/настройках подключения и откатить при необходимости.'], первый: log_agent


  [log_agent] → handoff → dba_agent


  [dba_agent] Решено → реплан


  [Планировщик] План: ['Проверить логи API и БД на точную причину ошибки подключения к PostgreSQL', 'Проверить доступность PostgreSQL, параметры подключения и состояние сети/аутентификации', 'Если проблема не в инфраструктуре, проверить код и конфигурацию инициализации соединения'], первый: log_agent

Журнал (2 записей):
  [log_agent]: HANDOFF dba_agent
  [dba_agent]: Похоже, API падает на этапе подключения к PostgreSQL: проверьте доступность сервера, параметры conne

Решение: Похоже, API падает на этапе подключения к PostgreSQL: проверьте доступность сервера, параметры connection string, DNS/сеть и лимиты соединений в БД. Быстрое восстановление — убедиться, что PostgreSQL 


## Итоги

**Network + Plan-Execute** сочетает централизованное планирование с децентрализованным исполнением:

- **Plan-Execute** даёт структуру: планировщик создаёт план расследования и контролирует прогресс через реплан
- **Network** даёт гибкость: агенты передают задачи друг другу напрямую через `Command(goto=...)`, минуя планировщик, когда следующий шаг очевиден
- **Защита от зацикливания**: лимит записей в журнале и проверка `resolution` гарантируют завершение

Эта комбинация идеальна для сценариев, где стратегия требует централизации (общий план), но тактические решения эффективнее на месте (агент логов видит ошибку БД и сразу передаёт DBA).